# PenG — AI Học Tập
Clone, install, and test PenG from Google Colab with GPU T4.

**Prerequisites:** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Clone repo
!git clone https://github.com/canhcutlo/PenG.git
%cd PenG

In [ ]:
# 2. Check GPU
!nvidia-smi

In [ ]:
# 3. Install dependencies
# Pillow>=10.0,<11 for Surya-ocr compatibility on Python 3.12
!pip install -r requirements-colab.txt
!pip install pyngrok nest-asyncio

In [ ]:
# 4. Compile check
!python -m compileall app

In [ ]:
# 5. Run unit tests (skip AI model integration tests)
!pytest tests/ -v -m "not integration"

In [ ]:
# 6. Start FastAPI server in background + open ngrok + health check
import subprocess, sys, time, requests
from pyngrok import ngrok

# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Start uvicorn in a detached subprocess (no shell blocking)
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait for uvicorn to boot (up to 10s)
for _ in range(20):
    time.sleep(0.5)
    try:
        r = requests.get("http://localhost:8000/api/health", timeout=1)
        if r.status_code == 200:
            print("Server started OK")
            break
    except requests.RequestException:
        pass
else:
    print("WARNING: Server may not have started; check manually")

# Open ngrok tunnel — use .public_url for the actual string
tunnel = ngrok.connect(8000)
url = tunnel.public_url

print(f"\n=== PenG is running ===")
print(f"Frontend: {url}")
print(f"API docs: {url}/docs")
print(f"Health:   {url}/api/health")
print(f"========================")

## Verification checklist

Mở `public_url` trong trình duyệt và kiểm tra từng mục:

- [ ] **Health**: `/api/health` → `{"status":"ok","db":"ok"}`
- [ ] **Frontend**: `/` → hiện 5 tabs (Upload / Hỏi đáp / Quiz / Mindmap / Lịch sử)
- [ ] **Upload**: Kéo-thả file ảnh nhỏ → trả `doc_id` + `job_id`; poll `/api/jobs/{job_id}`
- [ ] **Query**: Nhập câu hỏi vào tab Hỏi đáp → trả answer
- [ ] **Mindmap**: Nhập `doc_id` vào tab Mindmap → hiển thị markmap
- [ ] **Quiz**: Generate quiz từ `doc_id` → hiển thị câu hỏi → submit → chấm điểm
- [ ] **History**: Tab Lịch sử hiển thị hoạt động đã log

## Troubleshooting

| Vấn đề | Cách sửa |
|---|---|
| `ModuleNotFoundError: surya` | Colab Python 3.12: chạy `!pip install "Pillow>=10.0,<11" && !pip install surya-ocr` |
| CUDA out of memory | Tải Qwen/Llama với 4-bit quantization; hoặc dùng CPU |
| ngrok không kết nối | Kiểm tra token tại https://dashboard.ngrok.com |
| Server không khởi động | Chạy lại cell 6 || Muốn dừng server | Chạy `proc.terminate()` hoặc Restart Runtime |